ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, we use K-Means clustering to segment credit card customers based on their usage behavior.
This is an **unsupervised learning** problem because the dataset contains no target label.


## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Get the Data

**Read the `CC_GENERAL.csv` file.**

In [1]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows.**

In [1]:
df.head()

**Check the shape.**

In [1]:
print(df.shape)

(8950, 18)


**Check info.**

In [1]:
df.info()

**Check summary statistics.**

In [1]:
df.describe()

## Data Cleaning

The `CUST_ID` column is only an identifier and carries no behavioral information — including it would distort the distance calculations K-Means relies on.

**Drop `CUST_ID`.**

In [1]:
df.drop('CUST_ID', axis=1, inplace=True)

**Check missing values.**

In [1]:
df.isnull().sum()

CREDIT_LIMIT        1
MINIMUM_PAYMENTS    313
dtype: int64 (all others 0)


Two columns have missing values:
- `CREDIT_LIMIT` — **1 missing value**
- `MINIMUM_PAYMENTS` — **313 missing values**

We use **mean imputation** to fill them so we retain all 8 950 rows.

**Fill missing values with the column mean.**

In [1]:
df.fillna(df.mean(), inplace=True)

**Verify no missing values remain.**

In [1]:
df.isnull().sum().sum()

0


## Exploratory Data Analysis

### Histograms — Understanding Feature Distributions

Most financial features (BALANCE, PURCHASES, CASH_ADVANCE, CREDIT_LIMIT) are **right-skewed**: 
the majority of customers have low values but a few have very high values.  
Frequency columns (0–1 range) show more uniform or bimodal distributions.


In [1]:
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].tick_params(labelsize=7)
for j in range(len(df.columns), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Correlation Heatmap

Key observations:
- **PURCHASES**, **ONEOFF_PURCHASES**, and **INSTALLMENTS_PURCHASES** are highly correlated with each other and with **PURCHASES_TRX**
- **CASH_ADVANCE** is highly correlated with **CASH_ADVANCE_TRX** and **CASH_ADVANCE_FREQUENCY**
- **CREDIT_LIMIT** and **PAYMENTS** show moderate positive correlation


In [1]:
plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(df.corr(), dtype=bool))
sns.heatmap(df.corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, annot_kws={'size': 6.5})
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Scatter Plot: Balance vs Purchases

In [1]:
plt.figure(figsize=(7, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.3, s=8, color='steelblue')
plt.xlabel('BALANCE'); plt.ylabel('PURCHASES')
plt.title('Balance vs Purchases')
plt.tight_layout(); plt.show()

### Scatter Plot: Balance vs Cash Advance

In [1]:
plt.figure(figsize=(7, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.3, s=8, color='tomato')
plt.xlabel('BALANCE'); plt.ylabel('CASH_ADVANCE')
plt.title('Balance vs Cash Advance')
plt.tight_layout(); plt.show()

## Feature Scaling

K-Means uses **Euclidean distance**. Without scaling, features with large ranges (e.g. BALANCE up to ~19 000)
would completely dominate features with small ranges (e.g. frequency columns 0–1).
StandardScaler transforms each feature to zero mean and unit variance.


In [1]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print("Scaled shape:", X_scaled.shape)

Scaled shape: (8950, 17)


## Elbow Method

In [1]:
inertia_values = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia_values.append(km.inertia_)
print("Inertia values computed for K = 1 … 10")

Inertia values computed for K = 1 … 10


In [1]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, 11), inertia_values, 'o-', color='steelblue', lw=2, ms=8)
plt.axvline(x=4, color='tomato', ls='--', alpha=0.8, label='Elbow ≈ K=4')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (WCSS)', fontsize=12)
plt.title('Elbow Method', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Elbow Interpretation**

The curve shows a noticeable bend around **K = 3 or K = 4** — after this point, 
the reduction in inertia becomes much smaller. This suggests **K = 4** as a reasonable choice,
balancing compactness with interpretability.


## Silhouette Score

In [1]:
silhouette_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels, sample_size=2000, random_state=42)
    silhouette_scores.append(score)
print("Silhouette scores computed for K = 2 … 10")

Silhouette scores computed for K = 2 … 10


In [1]:
plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), silhouette_scores, 's-', color='seagreen', lw=2, ms=8)
plt.axvline(x=4, color='tomato', ls='--', alpha=0.8, label='Chosen K=4')
plt.xlabel('K', fontsize=12); plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score by K', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [1]:
sil_table = pd.DataFrame({'K': range(2, 11),
                          'Silhouette Score': [round(s, 4) for s in silhouette_scores]})
print(sil_table.to_string(index=False))

 K  Silhouette Score
 2            0.2157
 3            0.2383
 4            0.2003
 5            0.1976
 6            0.2051
 7            0.2166
 8            0.2102
 9            0.2177
10            0.2216


**Silhouette Interpretation**

The highest silhouette score occurs at **K = 3** (0.2383), and K = 2 scores second.  
However, K = 3 may be too coarse for business segmentation.  
We choose **K = 4** which combines a reasonable silhouette (0.2003) with the elbow result
and produces **4 distinct, interpretable customer segments**.


## Create the Final K-Means Model (K = 4)

In [1]:
final_km = KMeans(n_clusters=4, random_state=42, n_init=10)
final_km.fit(X_scaled)
print("Model trained. Inertia:", round(final_km.inertia_, 2))

Model trained. Inertia: 72441.61


In [1]:
df['Cluster'] = final_km.labels_

In [1]:
df.head()

## Cluster Analysis

In [1]:
cluster_summary = df.groupby('Cluster').mean().round(2)
cluster_summary

In [1]:
cluster_counts = df['Cluster'].value_counts().sort_index()
print("Customers per cluster:")
print(cluster_counts)

Customers per cluster:
Cluster
0    3367
1     409
2    1198
3    3976
Name: count, dtype: int64


## Visualizing the Final Clusters (PCA)

PCA reduces 17 dimensions to 2 for visualization. The clusters are performed in the original 17D space — PCA is only used to plot them.


In [1]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
ev = pca.explained_variance_ratio_
print(f"Explained variance: PC1={ev[0]*100:.1f}%  PC2={ev[1]*100:.1f}%  Total={sum(ev)*100:.1f}%")

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
plt.figure(figsize=(10, 7))
for cl in range(4):
    m = df['Cluster'] == cl
    plt.scatter(pca_coords[m, 0], pca_coords[m, 1],
                s=12, alpha=0.5, color=colors[cl], label=f'Cluster {cl} (n={m.sum()})')
plt.xlabel(f'PC1 ({ev[0]*100:.1f}% variance)', fontsize=11)
plt.ylabel(f'PC2 ({ev[1]*100:.1f}% variance)', fontsize=11)
plt.title('K-Means Clusters (K=4) — PCA 2D View', fontsize=13, fontweight='bold')
plt.legend(fontsize=10); plt.grid(alpha=0.2)
plt.tight_layout(); plt.show()

Explained variance: PC1=26.5%  PC2=12.3%  Total=38.8%


## Final Questions

### 1. Why is this an unsupervised learning problem?
Because the dataset has **no target column** (no predefined customer group labels). 
The algorithm must discover patterns and structure in the data on its own, without guidance from labeled examples.

### 2. Why did we remove the `CUST_ID` column?
`CUST_ID` is only an identifier — it carries no behavioral information about the customer.
Including it would introduce meaningless distances in the clustering computation and distort the results.

### 3. Which columns had missing values?
- `CREDIT_LIMIT` — 1 missing value  
- `MINIMUM_PAYMENTS` — 313 missing values

### 4. How did you handle the missing values?
We used **mean imputation**: each missing value was replaced with the mean of its column.
This approach preserves all 8,950 rows and does not bias the distribution significantly.

### 5. Why is scaling important before applying K-Means?
K-Means measures distance between points. Without scaling, features with large ranges 
(e.g. BALANCE ≈ 0–19,000) would dominate features with small ranges (e.g. frequency columns ≈ 0–1).
StandardScaler ensures every feature contributes equally to the distance calculation.

### 6. Which K value did you choose? Explain.
We chose **K = 4** based on two methods:
- **Elbow method**: The inertia curve shows a clear bend around K = 3–4; after K = 4 the reduction becomes marginal.
- **Silhouette score**: K = 3 gives the highest score (0.2383), but K = 4 (0.2003) produces more granular, business-meaningful segments and aligns with the elbow. Four segments is also a practical number for a marketing strategy.

### 7. Describe each customer segment:

| Cluster | Size | Profile |
|---------|------|---------|
| **Cluster 0** | 3,367 | **Active Purchasers** — moderate balance, high purchase frequency, low cash advance. Regular shoppers who make both one-off and installment purchases. |
| **Cluster 1** | 409 | **High-Value / VIP Customers** — highest purchases (avg $7,682), highest credit limit ($9,697), highest payments. They spend a lot and pay off regularly. |
| **Cluster 2** | 1,198 | **Cash Advance Reliant** — highest cash advance (avg $4,521), very high cash advance frequency (0.48) and transactions (14.28). Low purchases. These customers use the card mainly as a loan source. |
| **Cluster 3** | 3,976 | **Low Activity / Dormant** — lowest purchases ($270), lowest purchase frequency (0.17). Low balances and credit limits. Minimal card usage overall. |

### 8. Which cluster may represent high-value customers?
**Cluster 1** — they have the highest average purchases ($7,682), highest credit limit ($9,697), and highest payments ($7,289). These are the most profitable customers.

### 9. Which cluster may represent customers who rely more on cash advance?
**Cluster 2** — with the highest average cash advance ($4,521), highest cash advance frequency (0.48), and highest number of cash advance transactions (14.28). These customers use the card primarily for cash withdrawal rather than purchases.

### 10. How can a company use these clusters for marketing strategy?

| Cluster | Strategy |
|---------|----------|
| **Cluster 0 — Active Purchasers** | Offer loyalty rewards and cashback programs to increase purchase volume and retention. |
| **Cluster 1 — VIP Customers** | Provide premium card upgrades, exclusive benefits, travel rewards, and dedicated customer service. |
| **Cluster 2 — Cash Advance Users** | Offer lower interest rates or personal loan products to meet their credit needs more efficiently, reducing default risk. |
| **Cluster 3 — Dormant Customers** | Send re-engagement campaigns with special offers, spending incentives, or simplified rewards to activate card usage. |
